# Notebook 20 — Gephi Export

Exports node and edge CSVs for three network layers: **Arms**, **ODA**, **Colonial**.

## Outputs (`outputs/gephi/`)

| File | Content | Use |
|------|---------|-----|
| `nodes_attributes.csv` | One row per country, all attributes | Base node table for all Gephi projects |
| `edges_arms_2007.csv` | Arms flows, 2007 only | **Snapshot**: peak IRQ killing period / IRQ blind spot |
| `nodes_2007_arms.csv` | Nodes present in 2007 arms network | Pair with above |
| `edges_oda_2010.csv` | ODA flows, 2010 only | **Snapshot**: USA→PHL post-Maguindanao |
| `nodes_2010_oda.csv` | Nodes present in 2010 ODA network | Pair with above |
| `edges_colonial_static.csv` | All colonial ties (time-invariant) | **Snapshot**: full colonial structure |
| `nodes_colonial.csv` | Nodes with any colonial tie | Pair with above |
| `edges_arms_allyears.csv` | Arms flows 1992–2024, Year column | **Animation** |
| `edges_oda_allyears.csv` | ODA flows 1992–2024, Year column | **Animation** |

## Case study logic
- **IRQ** (`case_study='IRQ'`): `colonial_tie=1` fires on GBR (British Mandate 1920–32), not USA (2003 invasion). Arms 2007 snapshot shows USA as the dominant hub sender during the CPJ killing peak (2006–2008) — but COLDAT cannot encode this as a colonial relationship. That's the blind spot made structural.
- **PHL** (`case_study='PHL'`): `colonial_tie=1` fires on ESP only in COLDAT (Spanish period 1565–1898; US colonial period 1898–1946 is absent — see COLDAT coverage note in Section 5). ODA 2010 snapshot shows USA→PHL channel active immediately after the 2009 Maguindanao massacre (34 journalists killed). Consistent with the oda_x_colonial mechanism (NegBin p=0.003).

`case_study` is a node attribute — use it in Gephi to partition node colour (IRQ=red, PHL=orange, others=grey).

---

## Gephi import instructions

**For each project:**
1. File → New Project
2. Data Laboratory → Import Spreadsheet → `nodes_*.csv` → select **Nodes** table → set `Id` column
3. Data Laboratory → Import Spreadsheet → `edges_*.csv` → select **Edges** table → set Source/Target/Weight → **Directed** graph

**Layout**: Overview tab → ForceAtlas2 → enable LinLog mode + Prevent Overlap → Run ~2000 iterations → Stop

**Node size** (Appearance → Nodes → Size icon → Ranking): map `journalist_killings_total`, min=5, max=40

**Node colour** (Appearance → Nodes → Fill colour → Partition):
- Audit figures: partition by `case_study` (IRQ / PHL / blank)
- Colonial figure: partition by `colonial_tie_flag`

**Edge weight** (Appearance → Edges → Size → Ranking): map `Weight`

**Labels**: show only nodes where `journalist_killings_total > 10` — Data Laboratory → right-click column → Copy data to other column → Label, then filter in Overview

**Animation** (arms/ODA all-years files): Tools → Timeline → enable dynamic attributes → set `Year` as time axis → Play 1992→2024. Export via Preview → SVG or use screen record for video.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../data/merged')
OUTPUT_DIR  = Path('../outputs/gephi')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the oda-capped dyadic panel (natural units, post-floor — correct for edge weights)
DYADIC_PATH  = DATA_DIR / 'dyadic_panel_1992_2024_oda_capped.csv'
MONADIC_PATH = DATA_DIR / 'panel_monadic_1992_2024.csv'

print('Loading panels...')
dyadic  = pd.read_csv(DYADIC_PATH)
monadic = pd.read_csv(MONADIC_PATH)

print(f'Dyadic  shape: {dyadic.shape}')
print(f'Monadic shape: {monadic.shape}')
print(f'\nDyadic cols:  {dyadic.columns.tolist()}')
print(f'Monadic cols: {monadic.columns.tolist()}')

Loading panels...
Dyadic  shape: (115640, 14)
Monadic shape: (6358, 13)

Dyadic cols:  ['sender_iso3', 'recipient_iso3', 'year', 'arms_tiv', 'bilateral_oda', 'econ_neocol_score', 'colonial_tie', 'journalist_killings', 'gdp_per_capita', 'gdp_per_capita_log', 'population', 'population_log', 'armed_conflict', 'conflict_intensity']
Monadic cols: ['recipient_iso3', 'year', 'arms_tiv_total', 'oda_total', 'econ_neocol_score_total', 'colonial_tie_flag', 'journalist_killings', 'gdp_per_capita', 'gdp_per_capita_log', 'population', 'population_log', 'armed_conflict', 'conflict_intensity']


## 1. Node attribute table

One row per country. Built from the monadic panel (recipient countries with killing data).
Sender-only countries (Global North arms/ODA exporters that never appear as recipients)
are added with `killings=0` so Gephi can draw edges from them and show the hub structure.

In [ ]:
# ── Recipient nodes from monadic panel ────────────────────────────────────
recipient_nodes = monadic.groupby('recipient_iso3').agg(
    journalist_killings_total = ('journalist_killings', 'sum'),
    colonial_tie_flag         = ('colonial_tie_flag', 'max'),
    armed_conflict_pct        = ('armed_conflict', 'mean'),
    gdp_per_capita_log_mean   = ('gdp_per_capita_log', 'mean'),
    population_log_mean       = ('population_log', 'mean'),
).reset_index().rename(columns={'recipient_iso3': 'Id'})

# ── Sender-only nodes ─────────────────────────────────────────────────────
# Countries that send arms/ODA but don't appear as recipients.
# Needed so Gephi can draw edges from them — they're the hub nodes.
all_senders  = set(dyadic['sender_iso3'].unique())
existing_ids = set(recipient_nodes['Id'])
sender_only  = sorted(all_senders - existing_ids)

sender_nodes = pd.DataFrame({
    'Id':                        sender_only,
    'journalist_killings_total': 0,
    'colonial_tie_flag':         0,
    'armed_conflict_pct':        np.nan,
    'gdp_per_capita_log_mean':   np.nan,
    'population_log_mean':       np.nan,
})

nodes = pd.concat([recipient_nodes, sender_nodes], ignore_index=True)

# ── Case study flag ────────────────────────────────────────────────────────
nodes['case_study'] = nodes['Id'].map({'IRQ': 'IRQ', 'PHL': 'PHL'}).fillna('')
# Re-apply after concat to catch any NaNs introduced during concatenation
nodes['case_study'] = nodes['case_study'].fillna('')

# ── Gephi requires Label column ───────────────────────────────────────────
nodes['Label'] = nodes['Id']

col_order = ['Id', 'Label', 'case_study', 'journalist_killings_total',
             'colonial_tie_flag', 'armed_conflict_pct',
             'gdp_per_capita_log_mean', 'population_log_mean']
nodes = nodes[col_order]

print(f'Total nodes:          {len(nodes)}')
print(f'  Recipient nodes:    {len(recipient_nodes)}')
print(f'  Sender-only nodes:  {len(sender_nodes)}')
print(f'  Nodes with killings > 0: {(nodes.journalist_killings_total > 0).sum()}')
print(f'  colonial_tie_flag=1: {(nodes.colonial_tie_flag == 1).sum()}')
print(f'\nCase study countries:')
print(nodes[nodes['case_study'] != ''][['Id', 'journalist_killings_total', 'colonial_tie_flag', 'case_study']])
print(f'\ncase_study NaN check: {nodes["case_study"].isna().sum()} NaNs (should be 0)')

In [3]:
# Save master node table
out = OUTPUT_DIR / 'nodes_attributes.csv'
nodes.to_csv(out, index=False)
print(f'Saved: {out}  ({len(nodes)} rows)')

Saved: ../outputs/gephi/nodes_attributes.csv  (215 rows)


## 2. Edge builder helpers

In [4]:
def build_edges(df, weight_col, layer_name, year=None, min_weight=0):
    """
    Build a Gephi-ready directed edge table from the dyadic panel.

    Parameters
    ----------
    df          : dyadic panel DataFrame
    weight_col  : column to use as edge weight ('arms_tiv' or 'bilateral_oda')
    layer_name  : string label for the Layer column
    year        : int or None — if int, filter to that year only (snapshot mode)
    min_weight  : drop rows at or below this weight (default 0)

    Returns
    -------
    DataFrame: Source, Target, Weight, Type, Year, Layer
    """
    sub = df.copy()
    if year is not None:
        sub = sub[sub['year'] == year]

    edges = (
        sub[sub[weight_col] > min_weight]
        [['sender_iso3', 'recipient_iso3', 'year', weight_col]]
        .rename(columns={
            'sender_iso3':    'Source',
            'recipient_iso3': 'Target',
            'year':           'Year',
            weight_col:       'Weight',
        })
        .copy()
    )
    edges['Type']  = 'Directed'
    edges['Layer'] = layer_name
    return edges.reset_index(drop=True)


def nodes_for_edges(edge_df, nodes_df):
    """
    Return the subset of nodes_df that appear in edge_df (as Source or Target).
    Keeps Gephi import clean — no orphan nodes in snapshot projects.
    """
    active = set(edge_df['Source']) | set(edge_df['Target'])
    return nodes_df[nodes_df['Id'].isin(active)].copy()


print('Helpers defined.')

Helpers defined.


## 3. Snapshot — Arms 2007 (IRQ blind spot figure)

2007 = peak of the Iraqi killing period (CPJ worst years 2006–2008). USA is the dominant arms sender; IRQ is the most exposed spoke. This is the visual argument for the blind spot — the arms channel captures the military neo-colonial relationship that COLDAT cannot encode.

Note: SIPRI records zero arms transfers to IRQ in 2003 — the invasion used pre-positioned equipment. Arms transfers begin 2004 as USA rebuilds Iraqi security forces (USA→IRQ: 2007 ~160 TIV, 2008 ~275 TIV, peak 889 in 2016). The 2003 snapshot would show an empty IRQ node, misrepresenting the military dependency structure during the killing period.

Expected top senders: USA, RUS, GBR, FRA, DEU
Expected IRQ inflow: USA dominant

In [ ]:
edges_arms_2007 = build_edges(dyadic, 'arms_tiv', 'Arms', year=2007)

print(f'Arms edges 2007:   {len(edges_arms_2007)}')
print(f'Unique senders:    {edges_arms_2007.Source.nunique()}')
print(f'Unique recipients: {edges_arms_2007.Target.nunique()}')

print('\nTop 10 edges by TIV:')
print(edges_arms_2007.nlargest(10, 'Weight')[['Source', 'Target', 'Weight']].to_string(index=False))

print('\nAll IRQ inflows (case study — expect USA dominant):')
print(edges_arms_2007[edges_arms_2007['Target'] == 'IRQ']
      .sort_values('Weight', ascending=False)[['Source', 'Target', 'Weight']].to_string(index=False))

# Save
edges_arms_2007.to_csv(OUTPUT_DIR / 'edges_arms_2007.csv', index=False)
nodes_for_edges(edges_arms_2007, nodes).to_csv(OUTPUT_DIR / 'nodes_2007_arms.csv', index=False)
print(f'\nSaved: edges_arms_2007.csv  +  nodes_2007_arms.csv')

## 4. Snapshot — ODA 2010 (PHL donor-cover figure)

2010 = year immediately after the 2009 Maguindanao massacre (34 journalists killed;
single largest event in the CPJ dataset). USA→PHL ODA channel remains active,
consistent with the donor-cover mechanism: oda_x_colonial significant (NegBin p=0.003),
colonial_tie fires correctly on USA for Philippines.

Expected top donors: USA, JPN, DEU, GBR, AUS
Expected PHL inflows: USA dominant

In [6]:
edges_oda_2010 = build_edges(dyadic, 'bilateral_oda', 'ODA', year=2010)

print(f'ODA edges 2010:    {len(edges_oda_2010)}')
print(f'Unique donors:     {edges_oda_2010.Source.nunique()}')
print(f'Unique recipients: {edges_oda_2010.Target.nunique()}')

print('\nTop 10 edges by ODA (USD):')
print(edges_oda_2010.nlargest(10, 'Weight')[['Source', 'Target', 'Weight']].to_string(index=False))

print('\nPHL inflows (case study — expect USA top):')
print(edges_oda_2010[edges_oda_2010['Target'] == 'PHL']
      .nlargest(5, 'Weight')[['Source', 'Target', 'Weight']].to_string(index=False))

print('\nIRQ inflows 2010 (comparison — post-invasion ODA):')
print(edges_oda_2010[edges_oda_2010['Target'] == 'IRQ']
      .nlargest(5, 'Weight')[['Source', 'Target', 'Weight']].to_string(index=False))

# Save
edges_oda_2010.to_csv(OUTPUT_DIR / 'edges_oda_2010.csv', index=False)
nodes_for_edges(edges_oda_2010, nodes).to_csv(OUTPUT_DIR / 'nodes_2010_oda.csv', index=False)
print(f'\nSaved: edges_oda_2010.csv  +  nodes_2010_oda.csv')

ODA edges 2010:    3121
Unique donors:     37
Unique recipients: 151

Top 10 edges by ODA (USD):
Source Target  Weight
   USA    AFG 2938.84
   USA    IRQ 1609.61
   USA    PAK 1200.45
   USA    HTI 1077.90
   JPN    IND  981.14
   FRA    COG  909.40
   JPN    VNM  807.81
   USA    ETH  802.63
   JPN    AFG  745.66
   USA    PSE  714.61

PHL inflows (case study — expect USA top):
Source Target  Weight
   FRA    PHL  189.43
   USA    PHL  113.45
   AUS    PHL  106.17
   KOR    PHL   29.54
   ESP    PHL   27.01

IRQ inflows 2010 (comparison — post-invasion ODA):
Source Target  Weight
   USA    IRQ 1609.61
   JPN    IRQ  144.44
   AUS    IRQ   52.22
   TUR    IRQ   39.31
   DEU    IRQ   36.85

Saved: edges_oda_2010.csv  +  nodes_2010_oda.csv


## 5. Snapshot — Colonial structure (static audit figure)

Colonial ties are time-invariant in COLDAT. We deduplicate across years to get unique dyads.
This figure goes in the audit report to show the colonial tie structure across the sample.

Key things to check:
- IRQ → GBR (British Mandate 1920–32): confirms colonial_tie fires on the *wrong* patron
- PHL → USA (and possibly ESP): confirms colonial_tie fires on the *right* patron
- Top colonisers by number of colonies: GBR and FRA should dominate

In [7]:
# Deduplicate — colonial_tie=1 rows are repeated for every year in the dyadic panel
colonial_dyads = (
    dyadic[dyadic['colonial_tie'] == 1]
    [['sender_iso3', 'recipient_iso3']]
    .drop_duplicates()
    .rename(columns={
        'sender_iso3':    'Source',
        'recipient_iso3': 'Target',
    })
    .copy()
)
colonial_dyads['Weight'] = 1          # binary — all ties equal weight
colonial_dyads['Type']   = 'Directed'
colonial_dyads['Layer']  = 'Colonial'
colonial_dyads['Year']   = 'static'   # signals time-invariance in Gephi Data Laboratory

print(f'Colonial edges (unique dyads): {len(colonial_dyads)}')
print(f'Unique colonisers: {colonial_dyads.Source.nunique()}')
print(f'Unique colonies:   {colonial_dyads.Target.nunique()}')

print('\nColonisers by number of colonies (top 10):')
print(colonial_dyads['Source'].value_counts().head(10).to_string())

print('\nIRQ colonial ties (expect GBR — blind spot):')
print(colonial_dyads[colonial_dyads['Target'] == 'IRQ'].to_string(index=False))

print('\nPHL colonial ties (expect USA + possibly ESP):')
print(colonial_dyads[colonial_dyads['Target'] == 'PHL'].to_string(index=False))

# Save
colonial_dyads.to_csv(OUTPUT_DIR / 'edges_colonial_static.csv', index=False)
nodes_for_edges(colonial_dyads, nodes).to_csv(OUTPUT_DIR / 'nodes_colonial.csv', index=False)
print(f'\nSaved: edges_colonial_static.csv  +  nodes_colonial.csv')

Colonial edges (unique dyads): 160
Unique colonisers: 8
Unique colonies:   126

Colonisers by number of colonies (top 10):
Source
GBR    68
FRA    32
ESP    26
DEU    12
PRT    12
NLD     4
BEL     3
ITA     3

IRQ colonial ties (expect GBR — blind spot):
Source Target  Weight     Type    Layer   Year
   GBR    IRQ       1 Directed Colonial static

PHL colonial ties (expect USA + possibly ESP):
Source Target  Weight     Type    Layer   Year
   ESP    PHL       1 Directed Colonial static

Saved: edges_colonial_static.csv  +  nodes_colonial.csv


### COLDAT coverage note — PHL and IRQ
Audit of the colonial static export reveals two COLDAT encoding gaps relevant to the case studies:

**PHL**: COLDAT encodes only the Spanish colonial period (ESP→PHL, 1565–1898). The US colonial period (1898–1946) is absent — USA does not appear as a coloniser for PHL anywhere in COLDAT. This means the oda_x_colonial interaction for Philippines fires on a historical Spanish tie while the empirically active ODA donor is the United States. Same structural problem as Iraq, different century.

**IRQ**: COLDAT encodes GBR→IRQ (British Mandate 1920–32). USA does not appear. The 2003 invasion and subsequent occupation — the period driving 285 journalist killings — is structurally invisible to COLDAT because it has no category for post-1945 military occupations.

Both gaps should be documented in LIMITATIONS.md. The colonial layer Gephi figure should note in its caption that colonial_tie_flag=1 reflects 19th/early-20th century formal empire only.

## 6. Animation — Arms all years (1992–2024)

Full panel arms edge table. Gephi Timeline plugin reads the `Year` column.

Key moments to look for during animation:
- **1992–2000**: Soviet successor states emerge as major senders (RUS, UKR)
- **2001–2008**: US dominance peaks — AFG and IRQ become dominant receiving nodes
- **2010s**: Russia re-emerges (SYR, IND); France active in West Africa
- **IRQ node**: watch it swell 2003–2008 then shrink — visual of the blind spot period

In [8]:
edges_arms_all = build_edges(dyadic, 'arms_tiv', 'Arms')  # no year filter

print(f'Arms edges (all years): {len(edges_arms_all)}')
print(f'Years covered: {edges_arms_all.Year.min()} – {edges_arms_all.Year.max()}')
print(f'Unique senders:    {edges_arms_all.Source.nunique()}')
print(f'Unique recipients: {edges_arms_all.Target.nunique()}')

print('\nEdges per year (sample):')
print(edges_arms_all.groupby('Year').size().to_string())

print('\nIRQ total TIV inflow by year:')
irq = edges_arms_all[edges_arms_all['Target'] == 'IRQ'].groupby('Year')['Weight'].sum()
print(irq[irq > 0].to_string())

# Save
edges_arms_all.to_csv(OUTPUT_DIR / 'edges_arms_allyears.csv', index=False)
print(f'\nSaved: edges_arms_allyears.csv  ({len(edges_arms_all)} rows)')

Arms edges (all years): 12126
Years covered: 1992 – 2024
Unique senders:    108
Unique recipients: 178

Edges per year (sample):
Year
1992    279
1993    276
1994    308
1995    313
1996    310
1997    327
1998    308
1999    321
2000    309
2001    315
2002    320
2003    293
2004    319
2005    328
2006    363
2007    364
2008    366
2009    408
2010    423
2011    425
2012    425
2013    430
2014    438
2015    466
2016    461
2017    433
2018    426
2019    397
2020    374
2021    401
2022    401
2023    431
2024    368

IRQ total TIV inflow by year:
Year
2004      73.32
2005     193.35
2006     307.66
2007     299.42
2008     379.43
2009     417.86
2010     479.77
2011     622.00
2012     520.10
2013     373.14
2014     627.97
2015    1484.78
2016    1829.53
2017     944.94
2018     454.75
2019     143.98
2020       4.57
2022      54.60
2023      26.28
2024      22.87

Saved: edges_arms_allyears.csv  (12126 rows)


## 7. Animation — ODA all years (1992–2024)

Full panel ODA edge table.

Key channels to verify in the animation:
- **USA→PHL**: should be consistent across the full period; confirm 2009–2010 doesn't dip
- **FRA→CMR**: Cameroon case study channel — France as colonial donor
- **Post-conflict spikes**: ODA surges to IRQ post-2003, AFG post-2001 — shows reconstruction flows

In [9]:
edges_oda_all = build_edges(dyadic, 'bilateral_oda', 'ODA')  # no year filter

print(f'ODA edges (all years): {len(edges_oda_all)}')
print(f'Years covered: {edges_oda_all.Year.min()} – {edges_oda_all.Year.max()}')
print(f'Unique donors:     {edges_oda_all.Source.nunique()}')
print(f'Unique recipients: {edges_oda_all.Target.nunique()}')

print('\nUSA → PHL ODA by year (critical channel):')
usa_phl = (
    edges_oda_all[(edges_oda_all['Source'] == 'USA') & (edges_oda_all['Target'] == 'PHL')]
    .set_index('Year')['Weight']
    .sort_index()
)
print(usa_phl.to_string())

print('\nFRA → CMR ODA by year (Cameroon channel):')
fra_cmr = (
    edges_oda_all[(edges_oda_all['Source'] == 'FRA') & (edges_oda_all['Target'] == 'CMR')]
    .set_index('Year')['Weight']
    .sort_index()
)
print(fra_cmr.to_string())

# Save
edges_oda_all.to_csv(OUTPUT_DIR / 'edges_oda_allyears.csv', index=False)
print(f'\nSaved: edges_oda_allyears.csv  ({len(edges_oda_all)} rows)')

ODA edges (all years): 99125
Years covered: 1992 – 2024
Unique donors:     49
Unique recipients: 178

USA → PHL ODA by year (critical channel):
Year
1992    229.000000
1993    262.000000
1994    116.000000
1995    112.000000
1996     46.000000
1997     15.000000
1998     27.330000
1999     72.690000
2000     75.460000
2001     82.990000
2002     78.620000
2003     55.290000
2004     79.450000
2005     96.760000
2006     97.820000
2007     84.790000
2008     71.260000
2009     89.500000
2010    113.450000
2011    113.430000
2012    121.730000
2013    153.080000
2014    253.460000
2015    250.252675
2016    240.059300
2017    138.990740
2018    104.924830
2019    128.276320
2020    137.921910
2021    159.187890
2022    147.659662
2023    210.301133
2024    187.251085

FRA → CMR ODA by year (Cameroon channel):
Year
1992    407.270000
1993    425.710000
1994    308.550000
1995    265.160000
1996    176.010000
1997    199.840000
1998    152.770000
1999    134.800000
2000     86.220000
2001 

## 8. File manifest and Gephi import order

In [ ]:
print('=' * 65)
print('GEPHI EXPORT — FILE MANIFEST')
print('=' * 65)

files = sorted(OUTPUT_DIR.glob('*.csv'))
total_kb = 0
for f in files:
    kb = f.stat().st_size / 1024
    total_kb += kb
    print(f'  {f.name:<45}  {kb:>8.1f} KB')
print(f'\nTotal: {total_kb/1024:.2f} MB  |  {len(files)} files')

print('\n' + '=' * 65)
print('GEPHI PROJECT GUIDE')
print('=' * 65)
print('''
PROJECT A — Arms 2007 (IRQ blind spot, audit report figure)
  Nodes: nodes_2007_arms.csv
  Edges: edges_arms_2007.csv
  Size:  journalist_killings_total
  Colour (partition): case_study   → IRQ=red, PHL=orange, blank=grey

PROJECT B — ODA 2010 (PHL donor-cover, audit report figure)
  Nodes: nodes_2010_oda.csv
  Edges: edges_oda_2010.csv
  Size:  journalist_killings_total
  Colour (partition): case_study

PROJECT C — Colonial structure (time-invariant, report methods figure)
  Nodes: nodes_colonial.csv
  Edges: edges_colonial_static.csv
  Size:  journalist_killings_total
  Colour (partition): colonial_tie_flag  → 1=purple, 0=grey
  Note: IRQ→GBR edge = the blind spot. PHL→ESP only (USA absent from COLDAT).

PROJECT D — Animation: Arms 1992–2024
  Nodes: nodes_attributes.csv
  Edges: edges_arms_allyears.csv
  Tools → Timeline → Year → Play
  Watch: IRQ node swells 2004–2010; RUS re-emerges 2012+

PROJECT E — Animation: ODA 1992–2024
  Nodes: nodes_attributes.csv
  Edges: edges_oda_allyears.csv
  Tools → Timeline → Year → Play
  Watch: USA→PHL consistent; FRA→CMR persistent; IRQ post-2003 spike
''')

print('Case study node attributes (final check):')
print(nodes[nodes['case_study'] != ''][
    ['Id', 'journalist_killings_total', 'colonial_tie_flag', 'case_study']
].to_string(index=False))